In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from typing import Dict, List, Tuple
from data_loading import *
from loss_funcs import *

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}") # I have a 3090
    torch.set_float32_matmul_precision("medium")
    print("float32 matmul precision set to 'medium'")
# ── SARIMAX-specific imports ────────────────────────────────────
import itertools
import time
from collections import defaultdict, Counter

import mlflow
from tqdm.notebook import tqdm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter("ignore", ConvergenceWarning)

print(f"statsmodels SARIMAX ready, mlflow {mlflow.__version__}")

PyTorch version  : 2.10.0
CUDA available   : False
statsmodels SARIMAX ready, mlflow 3.11.1


In [2]:
weather_cols_all = ['temperature_2m',
       'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance']

other_cols = [ # these are not static
    'dam_price', 'buy_bm_price', 'sell_bm_price',
    'max_power', 'max_solar', 'max_ev'
]

cat_columns = [
    'eic_code', 'dso_desc', 'station_type', 'oblast',
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

time_cols = ['datetime', 'time_idx']

FUTURE_REALS = weather_cols_all + calendar_cols + static_cols + other_cols
print(f"y col is: {Y_COL}, group col is: {GROUP_COL}\n"
      f"features: {FUTURE_REALS}")

y col is: sum_of_kWh, group col is: eic_code
features: ['temperature_2m', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'Month', 'Day', 'Hour', 'day_of_week', 'season', 'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast', 'dam_price', 'buy_bm_price', 'sell_bm_price', 'max_power', 'max_solar', 'max_ev']


In [3]:
# this data has a data column and categorical columns are kept intact and will need to be handled.
# "time_idx" is already built in train, val and test and is continuous through them
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)
train = train[train["datetime"] >= "2025-06-01"].copy()

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train: {train.shape}")
print(f"val : {val.shape}")
print(f"test: {test.shape}")

Loading train …
Loading val   …
Loading test  …
train: (284400, 38)
val : (293880, 38)
test: (295430, 38)


In [4]:
# # If a model cant handle categorical columns natively, or through embedings use this data
# # It has no datetime column and all columns are numeric, as all cat column were ohe
# # GROUP_COL is the only exception, and is not ohe. Ohe it before training
# print("Loading train …")
# train = load_and_prepare(TRAIN_PATH_OHE)
#
# print("Loading val   …")
# val = load_and_prepare(VAL_PATH_OHE)
#
# print("Loading test  …")
# test = load_and_prepare(TEST_PATH_OHE)
#
# print(f"train: {train.shape}")
# print(f"val  : {val.shape}")
# print(f"test : {test.shape}")

In [5]:
# this cell samples locations, I'll use it if training takes too long, otherwise don't touch it

TARGET_STATIONS = 50

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

Stations: 50
Train rows : 36,000
Val rows   : 37,200
Test rows  : 37,397


In [6]:
training_cutoff = train["time_idx"].max()
val_cutoff      = val["time_idx"].max()
test_cutoff     = test["time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

training cutoff : 13125
val cutoff      : 13869
test cutoff     : 14617


In [7]:
# ────────────────────────────────────────────────────────────────
# SARIMAX configuration
#
# SARIMAX = Seasonal ARIMA with eXogenous regressors.
#   Non-seasonal order : (p, d, q)
#       p — autoregressive lags
#       d — non-seasonal differencing
#       q — moving-average lags
#   Seasonal order     : (P, D, Q, s)
#       P, D, Q — seasonal counterparts
#       s        — seasonal period (24 = daily, 168 = weekly for hourly data)
#
# We train ONE SARIMAX per eic_code (no cross-station mixing).
# Search strategy: AIC pre-filter on TRAIN-only fit, then re-rank the top-K
# candidates by money_pct on the validation horizon (the actual business loss).
# ────────────────────────────────────────────────────────────────

EXOG_COLS = [
    'dam_price', 'sell_bm_price', 'buy_bm_price',
    'Hour', 'day_of_week', 'Month',
]

# Grid kept compact on purpose — SARIMAX with s=168 is expensive, and we fit
# one model per (station, candidate). Expand only if you have time budget.
P_RANGE   = [0, 1, 2]
D_RANGE   = [0, 1]
Q_RANGE   = [0, 1, 2]
P_S_RANGE = [0, 1]
D_S_RANGE = [0, 1]
Q_S_RANGE = [0, 1]
S_RANGE   = [24]   # weekly s=168 is very slow; enable if your time budget allows

# Top-K AIC candidates re-ranked by money_pct on validation
TOP_K_BY_AIC = 5

# Recursive forecast block size (hours). Validation/test span ~1 month each;
# refitting state every BLOCK hours keeps it current without leaking the future.
FORECAST_BLOCK = 48

# Optional cap on stations actually optimised (None = all of them).
# If grid search is too slow on the full set, the remaining stations fall back
# to the most-frequent best order from the optimised subset.
MAX_STATIONS_TO_OPTIMISE = None

param_grid = list(itertools.product(
    P_RANGE, D_RANGE, Q_RANGE,
    P_S_RANGE, D_S_RANGE, Q_S_RANGE,
    S_RANGE,
))
# Drop the all-zeros configuration — it's just OLS on exog with no dynamics.
param_grid = [g for g in param_grid if not (g[0]==0 and g[1]==0 and g[2]==0
                                            and g[3]==0 and g[4]==0 and g[5]==0)]
print(f"SARIMAX grid size: {len(param_grid)} candidate configurations")
print(f"Exogenous variables ({len(EXOG_COLS)}): {EXOG_COLS}")

# ── MLflow run ──────────────────────────────────────────────────
mlflow.set_experiment("sarimax_electricity")
mlflow.start_run(run_name=f"sarimax_grid_{int(time.time())}")
mlflow.log_params({
    "model":            "SARIMAX",
    "y_col":            Y_COL,
    "group_col":        GROUP_COL,
    "exog_cols":        ",".join(EXOG_COLS),
    "grid_size":        len(param_grid),
    "top_k_by_aic":     TOP_K_BY_AIC,
    "forecast_block":   FORECAST_BLOCK,
    "seasonal_periods": ",".join(map(str, S_RANGE)),
    "n_train_rows":     len(train),
    "n_val_rows":       len(val),
    "n_test_rows":      len(test),
    "n_stations":       train[GROUP_COL].nunique(),
})
print(f"MLflow run started: {mlflow.active_run().info.run_id}")

SARIMAX grid size: 143 candidate configurations
Exogenous variables (6): ['dam_price', 'sell_bm_price', 'buy_bm_price', 'Hour', 'day_of_week', 'Month']
MLflow run started: 9152e97d18584a9795dc116f622f5cdb


In [8]:
# ────────────────────────────────────────────────────────────────
# Per-station data prep + recursive forecasting helpers
# ────────────────────────────────────────────────────────────────

def get_station_series(df: pd.DataFrame, eic: str):
    """Return (y, exog, full_group) for one station, sorted by time_idx.

    SARIMAX consumes:
      - endog : 1-D array of the target (sum_of_kWh)
      - exog  : 2-D array of exogenous regressors aligned 1:1 with endog
    The price columns + calendar features are known at forecast time
    (price feeds depend only on datetime, calendar features are
    deterministic), so they're safe as exogenous inputs — no leakage.
    """
    g = df[df[GROUP_COL] == eic].sort_values("time_idx")
    y    = g[Y_COL].to_numpy(dtype=float)
    exog = g[EXOG_COLS].to_numpy(dtype=float)
    return y, exog, g


def fit_sarimax(y, exog, order, seasonal_order):
    """Fit a SARIMAX with sensible defaults; return fitted result or None on failure."""
    try:
        model = SARIMAX(
            endog=y,
            exog=exog,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
            simple_differencing=False,
        )
        return model.fit(disp=False, maxiter=50, method="lbfgs")
    except Exception:
        return None


def recursive_forecast(fit_result, exog_future: np.ndarray,
                       block: int = FORECAST_BLOCK) -> np.ndarray:
    """Forecast `len(exog_future)` steps ahead in blocks of `block` hours.

    Each block uses model.get_forecast over its own exogenous slice, then the
    state is *appended* with the just-produced point forecasts so the next
    block continues from the correct internal state. We never feed back true
    target values from the held-out horizon — only the model's own predictions —
    so there is no leakage.
    """
    horizon = len(exog_future)
    preds   = np.empty(horizon, dtype=float)
    state   = fit_result
    cursor  = 0
    while cursor < horizon:
        step = min(block, horizon - cursor)
        exog_slice = exog_future[cursor:cursor + step]
        fc = state.get_forecast(steps=step, exog=exog_slice)
        block_pred = np.asarray(fc.predicted_mean, dtype=float)
        preds[cursor:cursor + step] = block_pred
        cursor += step
        if cursor < horizon:
            # Extend state with our own predictions (no future-target leakage).
            state = state.append(endog=block_pred, exog=exog_slice, refit=False)
    return preds


def _prices_arr(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values


print("SARIMAX helpers ready.")

SARIMAX helpers ready.


In [ ]:
# ────────────────────────────────────────────────────────────────
# Training: per-station SARIMAX grid search
#
#   For every eic_code:
#     1. fit each candidate (p,d,q)x(P,D,Q,s) on TRAIN only
#     2. keep TOP_K_BY_AIC by training-AIC (cheap pre-filter)
#     3. forecast the validation horizon recursively for each survivor
#     4. score each candidate's val forecast with money_pct (the business loss)
#     5. pick the best order — refit happens later in the inference cell
#
# We use AIC as a pre-filter because the full grid is large and most candidates
# are clearly worse statistically — re-ranking only the survivors with money_pct
# concentrates compute where it matters.
# ────────────────────────────────────────────────────────────────

stations_all = sorted(train[GROUP_COL].unique().tolist())
if MAX_STATIONS_TO_OPTIMISE is not None:
    stations_to_optimise = stations_all[:MAX_STATIONS_TO_OPTIMISE]
else:
    stations_to_optimise = stations_all

print(f"Stations to optimise: {len(stations_to_optimise)} / {len(stations_all)}")

best_orders: Dict[str, Dict] = {}   # eic -> {'order':..., 'seasonal_order':..., 'val_money_pct':..., 'aic':...}
fit_failures: List[str] = []

outer = tqdm(stations_to_optimise, desc="Stations", leave=True)
for eic in outer:
    y_tr, exog_tr, _       = get_station_series(train, eic)
    y_val, exog_val, val_g = get_station_series(val,   eic)
    if len(y_tr) < 2 * 168 or len(y_val) == 0:
        fit_failures.append(eic)
        continue

    # ── Stage 1: AIC pre-filter on train ──
    aic_results = []
    inner = tqdm(param_grid, desc=f"grid {eic[:8]}", leave=False)
    for (p, d, q, P, D, Q, s) in inner:
        order, seasonal_order = (p, d, q), (P, D, Q, s)
        res = fit_sarimax(y_tr, exog_tr, order, seasonal_order)
        if res is None or not np.isfinite(res.aic):
            continue
        aic_results.append((res.aic, order, seasonal_order, res))

    if not aic_results:
        fit_failures.append(eic)
        continue

    aic_results.sort(key=lambda t: t[0])
    candidates = aic_results[:TOP_K_BY_AIC]

    # ── Stage 2: re-rank survivors by money_pct on validation ──
    best = None
    val_prices = _prices_arr(val_g)
    for aic, order, seasonal_order, res in candidates:
        try:
            val_pred = recursive_forecast(res, exog_val)
        except Exception:
            continue
        if not np.all(np.isfinite(val_pred)):
            continue
        # money_pct is the business loss — lower is better.
        score = money_pct(y_val, val_pred, *val_prices)
        if best is None or score < best["val_money_pct"]:
            best = {
                "order":          order,
                "seasonal_order": seasonal_order,
                "val_money_pct":  float(score),
                "aic":            float(aic),
            }

    if best is None:
        fit_failures.append(eic)
        continue
    best_orders[eic] = best
    outer.set_postfix(best=str(best["order"]) + str(best["seasonal_order"]),
                      money_pct=f"{best['val_money_pct']:.2f}")

print(f"\nOptimised: {len(best_orders)}  |  failures: {len(fit_failures)}")

# ── Fallback for stations that weren't optimised (or failed): use the modal
#    best order from the optimised subset. SARIMAX shapes share a lot across
#    stations of the same type, so a shared-order fallback is reasonable. ──
if best_orders:
    order_counts = Counter(
        (b["order"], b["seasonal_order"]) for b in best_orders.values()
    )
    fallback_order, fallback_seasonal = order_counts.most_common(1)[0][0]
    print(f"Fallback order for un-optimised stations: {fallback_order} x {fallback_seasonal}")

    for eic in stations_all:
        if eic not in best_orders:
            best_orders[eic] = {
                "order":          fallback_order,
                "seasonal_order": fallback_seasonal,
                "val_money_pct":  float("nan"),
                "aic":            float("nan"),
                "fallback":       True,
            }

# Persist a compact summary of selected orders as an MLflow artifact.
orders_df = pd.DataFrame([
    {GROUP_COL:         k,
     "order":           str(v["order"]),
     "seasonal_order":  str(v["seasonal_order"]),
     "val_money_pct":   v["val_money_pct"],
     "aic":             v["aic"],
     "fallback":        v.get("fallback", False)}
    for k, v in best_orders.items()
])
orders_path = "sarimax_best_orders.csv"
orders_df.to_csv(orders_path, index=False)
mlflow.log_artifact(orders_path)
mlflow.log_metric("n_optimised",
                  len(best_orders) - sum(1 for v in best_orders.values()
                                         if v.get("fallback")))
mlflow.log_metric("n_fit_failures", len(fit_failures))
print(orders_df.head())

Stations to optimise: 50 / 50


Stations:   0%|          | 0/50 [00:00<?, ?it/s]

grid 62Z00085:   0%|          | 0/143 [00:00<?, ?it/s]

grid 62Z04946:   0%|          | 0/143 [00:00<?, ?it/s]

In [20]:
# ────────────────────────────────────────────────────────────────
# Inference: produce val_eval and test_eval
#
# For each station:
#   - val_eval  : refit on TRAIN, recursive-forecast over the val window
#   - test_eval : refit on TRAIN+VAL, recursive-forecast over the test window
#
# Both eval frames carry time_idx, prices, true Y_COL, and 'pred' so the
# existing rolling_eval / per_station_metrics / plot cells work unchanged.
# ────────────────────────────────────────────────────────────────

val_chunks: List[pd.DataFrame] = []
test_chunks: List[pd.DataFrame] = []

for eic in tqdm(stations_all, desc="Forecasting"):
    cfg = best_orders.get(eic)
    if cfg is None:
        continue
    order, seasonal_order = cfg["order"], cfg["seasonal_order"]

    y_tr,  exog_tr,  _      = get_station_series(train, eic)
    y_val, exog_val, val_g  = get_station_series(val,   eic)
    y_te,  exog_te,  test_g = get_station_series(test,  eic)

    # ── Validation forecast: fit on TRAIN, forecast val horizon ──
    res_tr = fit_sarimax(y_tr, exog_tr, order, seasonal_order)
    if res_tr is not None and len(y_val) > 0:
        try:
            val_pred = recursive_forecast(res_tr, exog_val)
            v = val_g[[GROUP_COL, "datetime", "time_idx", Y_COL,
                       "dam_price", "sell_bm_price", "buy_bm_price"]].copy()
            v["pred"] = np.clip(val_pred, a_min=0.0, a_max=None)
            val_chunks.append(v)
        except Exception:
            pass

    # ── Test forecast: refit on TRAIN+VAL, forecast test horizon ──
    if len(y_te) == 0:
        continue
    y_full    = np.concatenate([y_tr, y_val]) if len(y_val) else y_tr
    exog_full = np.vstack([exog_tr, exog_val]) if len(exog_val) else exog_tr
    res_full = fit_sarimax(y_full, exog_full, order, seasonal_order)
    if res_full is None:
        # Fall back to extending the train-only fit with val observations.
        if res_tr is None:
            continue
        try:
            res_full = res_tr.append(endog=y_val, exog=exog_val, refit=False)
        except Exception:
            continue
    try:
        test_pred = recursive_forecast(res_full, exog_te)
        t = test_g[[GROUP_COL, "datetime", "time_idx", Y_COL,
                    "dam_price", "sell_bm_price", "buy_bm_price"]].copy()
        t["pred"] = np.clip(test_pred, a_min=0.0, a_max=None)
        test_chunks.append(t)
    except Exception:
        continue

val_eval  = pd.concat(val_chunks,  ignore_index=True) if val_chunks  else pd.DataFrame()
test_eval = pd.concat(test_chunks, ignore_index=True) if test_chunks else pd.DataFrame()

# Prediction-bias diagnostic (sum of predictions − sum of actuals)
if not val_eval.empty:
    val_bias  = float(val_eval["pred"].sum()  - val_eval[Y_COL].sum())
    mlflow.log_metric("val_pred_bias",  val_bias)
if not test_eval.empty:
    test_bias = float(test_eval["pred"].sum() - test_eval[Y_COL].sum())
    mlflow.log_metric("test_pred_bias", test_bias)

# Save a small example-predictions artifact (one station, one week).
if not test_eval.empty:
    ex_eic = test_eval[GROUP_COL].iloc[0]
    ex = test_eval[test_eval[GROUP_COL] == ex_eic].head(168)
    ex_path = "sarimax_example_predictions.csv"
    ex.to_csv(ex_path, index=False)
    mlflow.log_artifact(ex_path)

print(f"val_eval : {val_eval.shape}")
print(f"test_eval: {test_eval.shape}")

Forecasting:   0%|          | 0/395 [00:00<?, ?it/s]

val_eval : (0, 0)
test_eval: (0, 0)


In [21]:
def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

val_smape_v     = smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))

test_smape_v     = smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")

mlflow.log_metrics({
    "val_smape":      val_smape_v,
    "val_rmse":       val_rmse_v,
    "val_mape":       val_mape_v,
    "val_money":      val_money_v,
    "val_money_pct":  val_money_pct_v,
    "test_smape":     test_smape_v,
    "test_rmse":      test_rmse_v,
    "test_mape":      test_mape_v,
    "test_money":     test_money_v,
    "test_money_pct": test_money_pct_v,
})
mlflow.end_run()
print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

KeyError: 'sum_of_kWh'

In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL:    grp,
            "n":          len(gdf),
            "SMAPE":      smape(gdf[Y_COL], gdf["pred"]),
            "RMSE":       rmse (gdf[Y_COL], gdf["pred"]),
            "MAE":        float(np.mean(np.abs(gdf[Y_COL].values - gdf["pred"].values))),
            "MAPE":       mape (gdf[Y_COL], gdf["pred"]),
            "MONEY":      money     (gdf[Y_COL], gdf["pred"], *_prices(gdf)),
            "MONEY_PCT":  money_pct (gdf[Y_COL], gdf["pred"], *_prices(gdf)),
            "BIAS":       float(gdf["pred"].sum() - gdf[Y_COL].sum()),
        })
    return pd.DataFrame(rows).sort_values("MONEY_PCT")


test_station_metrics = per_station_metrics(test_eval)
val_station_metrics  = per_station_metrics(val_eval)

# Aggregated cross-station metrics (mean over stations)
agg = {
    "agg_test_smape_mean":     float(test_station_metrics["SMAPE"].mean()),
    "agg_test_rmse_mean":      float(test_station_metrics["RMSE"].mean()),
    "agg_test_mae_mean":       float(test_station_metrics["MAE"].mean()),
    "agg_test_mape_mean":      float(test_station_metrics["MAPE"].mean()),
    "agg_test_money_pct_mean": float(test_station_metrics["MONEY_PCT"].mean()),
    "agg_test_money_sum":      float(test_station_metrics["MONEY"].sum()),
}
mlflow.log_metrics(agg)
test_station_metrics.to_csv("sarimax_test_per_station.csv", index=False)
mlflow.log_artifact("sarimax_test_per_station.csv")

print("Top-10 best stations (test MONEY_PCT):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test MONEY_PCT):")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):

    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}"
        f"  ({df['datetime'].min().date()} \u2013 {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()


# Example:
# plot_forecast(val_eval,  eic_code="<code>")
# plot_forecast(test_eval, eic_code="<code>", start_dt="2025-08-25", end_dt="2025-08-26")

In [ ]:
best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f"Best  station (MONEY_PCT): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST]  ")

print(f"Worst station (MONEY_PCT): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")